# Projet Recherche Opérationnelle

**Classe :** FISA A3 - Informatique  
**Groupe :** NOIRET Robin, LALOUANI Jalil, KLEBER Mattéo, CAPARROS Martin, UPJOHN Max  
**Période :** 03/2026 - 04/2026

---


# 1 - Contexte & Identification du problème

## 1.1 Cas d'application

Dans le cadre de l'appel à projets de l'**ADEME** sur la mobilité intelligente, notre bureau d'études **CesiCDP** a choisi de tester ses algorithmes d'optimisation sur un scénario concret et d'actualité : **la logistique de l'économie circulaire en milieu urbain dense**.

### Scénario expérimental
Un véhicule logistique part d'un centre de tri situé en périphérie (notre dépôt).</br> Sa mission quotidienne est double :
1. **Livrer** des équipements volumineux reconditionnés (ex: gros électroménager, pompes à chaleur) dans divers points relais ou chez des particuliers.

2. **Collecter** les anciens appareils en fin de vie (DEEE) pour les rapatrier au centre de tri afin d'y être recyclés.

### Intégration de nos contraintes de test
Pour valider la robustesse de notre modèle face à la réalité opérationnelle, notre environnement de simulation intègre deux contraintes physiques et réglementaires strictes :

* **Dépendances entre visites :** Il est physiquement impossible de charger un ancien équipement encombrant si le nouveau n'a pas encore été déchargé. L'algorithme doit donc garantir une **précédence** : la livraison d'un point A doit obligatoirement être effectuée avant la collecte au point B associé.</br></br>

* **La difficulté & l'inaccessibilité des routes :** Le réseau routier simulé intègre les nouvelles politiques d'urbanisme. L'algorithme devra trouver le trajet optimal en prenant en compte :
    * Des routes à coût majoré (zones de travaux urbains causant des détours ou des ralentissements).
    * Des routes strictement interdites (Zones à Faibles Émissions - ZFE inaccessibles, ou rues piétonnes/ponts avec restrictions de gabarit).

## 1.2 Identification du problème

### Périmètre de l'étude
L'étude se concentre sur **une tournée quotidienne**, réalisée par **un unique véhicule**, au départ et au retour d'**un dépôt unique**. Les aspects tels que la flotte multi-véhicules, la gestion des stocks au dépôt ou la planification sur plusieurs jours sont volontairement exclus du périmètre afin de concentrer l'analyse sur la structure combinatoire du routage sous contraintes.

### Reformulation du besoin
Derrière le scénario opérationnel décrit en 1.1 se cache un besoin algorithmique clair : **déterminer l'ordre de visite d'un ensemble de points d'arrêt** (livraisons et collectes) qui permette de réaliser l'intégralité de la tournée dans les meilleures conditions possibles, tout en respectant des règles de précédence et des restrictions de circulation imposées par le réseau routier.

### Objectif de l'étude
L'objectif d'optimisation retenu est la **minimisation de la distance totale parcourue** sur l'ensemble de la tournée. Ce critère sert de proxy direct aux enjeux soulevés par l'ADEME : réduction de la consommation énergétique, des émissions de gaz à effet de serre, et du temps de trajet du véhicule.

### Rattachement à un problème connu
Ce problème est une **variante du problème du voyageur de commerce** (*Travelling Salesman Problem*, TSP)[$^5$](#52-références-scientifiques) [$^6$](#52-références-scientifiques). La formulation classique du TSP, à savoir trouver le cycle hamiltonien de poids minimal dans un graphe pondéré correspond exactement à la structure de notre tournée. Les deux contraintes que nous ajoutons (arêtes interdites/coûteuses et précédences) en font une instance enrichie.

</br>

**Ce document pose les fondations théoriques de notre problème.** </br></br> La modélisation de ce problème de routage sous nos contraintes spécifiques choisies ainsi que la démonstration de son **explosion combinatoire** (NP-difficulté) justifieront, pour la suite du projet, l'implémentation de méthodes de résolution **approchées** capables de fournir des plans de tournée viables dans des temps de calcul optimisés.

# 2 - Définition mathématique du problème

## 2.1 Données du problème

On modélise le réseau routier par un graphe orienté pondéré :

$$G = (V, E, c')$$

avec :

- $V = \{v_0, v_1, \ldots, v_n\}$ : l'ensemble des **nœuds** (villes), où $v_0$ désigne le **dépôt** (point de départ et d'arrivée)
- $E = \{(i,j) \mid i \in V,\ j \in V,\ i \neq j\}$ : l'ensemble des **arêtes** (routes)
- $c'_{ij} = \alpha_{ij} \cdot c_{ij}$ : la **fonction de coût**, associant à chaque arête une distance. On ajoute une notion de coefficient (BigM)
> $\quad \quad \quad \alpha_{ij} = \begin{cases} 1 & \text{route classique} \\ [2,\ 10] & \text{route coûteuse} \\ [1000,\ 10000] & \text{route interdite} \end{cases}$

On définit également les ensembles suivants pour modéliser nos contraintes supplémentaires :

- $F \subseteq E$ : l'ensemble des **arêtes interdites** (routes bloquées, trop coûteuses ou inaccessibles)
- $D \subseteq V^2$ : l'ensemble des **paires de dépendance** de la forme $(i, j)$, signifiant que la ville $i$ doit être visitée avant la ville $j$ (livraison avant collecte)

## 2.2 Variables de décision

**Variables binaires de sélection des arêtes :**

$$x_{ij} \in \{0, 1\} \quad \forall (i,j) \in E$$

$x_{ij} = 1$ si et seulement si l'arête $(i,j)$ est empruntée dans la tournée, $0$ sinon.

**Variables de position (rang dans la tournée) :**

$$p_i \in \{1, \ldots, n\} \quad \forall i \in V \setminus \{v_0\}$$

$p_i$ représente le rang (ordre de passage) de la ville $i$ dans la tournée.

## 2.3 Fonction objectif

On cherche à **minimiser la distance totale** de la tournée :

$$\min \sum_{i \in V} \sum_{\substack{j \in V \\ j \neq i}} c'_{ij} \cdot x_{ij}$$

## 2.4 Contraintes

#### (C1) Chaque ville est visitée exactement une fois

$$\sum_{j \in V,\ j \neq i} x_{ij} = 1 \quad \forall i \in V  \quad \quad \quad

\sum_{i \in V,\ i \neq j} x_{ij} = 1 \quad \forall j \in V$$

#### (C2) Coût ou restriction de passage sur certaines arêtes

Certaines routes peuvent être dégradées (travaux, détours) ou totalement inaccessibles. Plutôt qu'une interdiction stricte, on modélise ces restrictions par un **coefficient de pénalité** $\alpha_{ij}$ appliqué au coût de chaque arête :

$$c'_{ij} = \alpha_{ij} \cdot c_{ij}$$

$$\alpha_{ij} = \begin{cases} 1 & \text{route classique} \\ [2,\ 10] & \text{route coûteuse} \\ [1000,\ 10000] & \text{route interdite} \end{cases}$$

> **Note :** Les arêtes "interdites" ne sont pas exclues par une contrainte dure, mais leur coût ajusté garantit qu'elles ne seront jamais empruntées par une solution optimale, étant donné l'échelle classique des coûts ($c_{ij} \in [1, 10]$).

#### (C3) Contraintes de dépendances de visite

Pour chaque paire de dépendance $(i, j) \in D$, la ville $i$ doit être visitée **avant** la ville $j$ :

$$p_i + 1 \leq p_j \quad \forall (i,j) \in D$$

> **Note :** Cette formule garantit un écart d'au moins une position entre les deux villes dans l'ordre de visite.

#### (C4) Élimination des sous-tours — Formulation MTZ

Pour éliminer les sous-tours (circuits ne passant pas par le dépôt), on utilise la formulation de Miller-Tucker-Zemlin (MTZ)[$^7$](#52-références-scientifiques) [$^8$](#52-références-scientifiques)  :

$$p_i - p_j + n \cdot x_{ij} \leq n - 1 \quad \forall (i,j) \in E,\ i \neq v_0,\ j \neq v_0$$

**Explication :**
- Si l'arête $(i,j)$ est utilisée ($x_{ij} = 1$) : la contrainte devient $p_i - p_j \leq -1$, soit $p_j \geq p_i + 1$. La ville $j$ est donc visitée **immédiatement après** la ville $i$ dans la tournée, ce sont donc deux villes consécutives. (cette contrainte représente l'élimination des sous-tours et non la dépendance de visite)
- Si l'arête $(i,j)$ n'est pas utilisée ($x_{ij} = 0$) : la contrainte devient $p_i - p_j \leq n - 1$, ce qui est toujours satisfait puisque les rangs appartiennent à $\{1, \dots, n\}$. La contrainte est relâchée.

## 2.5 Récapitulation (programmation linéaire)

On formule le problème comme un **programme linéaire en nombres entiers** (*Integer Linear Programming*, ILP). Les variables $x_{ij}$ sont binaires et les variables $p_i$ sont entières. Toutes les contraintes sont linéaires en ces variables.

$$
\text{PL} : \begin{cases}
\min & \displaystyle\sum_{i \in V} \sum_{\substack{j \in V \\ j \neq i}} c'_{ij} \cdot x_{ij} \\[10pt]
\ & \displaystyle\sum_{j \in V,\, j \neq i} x_{ij} = 1 \quad \forall i \in V \quad \text{(C1a)} \\[10pt]
& \displaystyle\sum_{i \in V,\, i \neq j} x_{ij} = 1 \quad \forall j \in V \quad \text{(C1b)} \\[10pt]
& p_i + 1 \leq p_j \quad \forall (i,j) \in D \quad \text{(C3)} \\[10pt]
& p_i - p_j + n \cdot x_{ij} \leq n - 1 \quad \forall (i,j) \in E,\ i \neq v_0,\ j \neq v_0 \quad \text{(C4)} \\[10pt]
& x_{ij} \in \{0, 1\} \\[10pt]
& p_i \in \{1, \ldots, n\} \\[10pt]
\end{cases}
$$

# 3 - Analyse de la complexité

La justification de la complexité de notre problème est primordiale pour orienter nos futurs choix d'implémentation. Pour ce faire, nous allons procéder par étapes, en partant d'un problème fondamental de la théorie des graphes pour arriver à notre cas d'usage logistique.

## 3.1. Le point de départ : Le problème du Cycle Hamiltonien (HC)

Le problème du Cycle Hamiltonien consiste à déterminer si, dans un graphe donné, il existe un cycle passant par chaque sommet exactement une fois.Ce problème de décision a été formellement classé comme NP-complet par Richard M. Karp en 1972 (il fait partie de ses 21 problèmes originaux). Cela signifie qu'il n'existe, à ce jour, aucun algorithme capable de le résoudre en un temps polynomial.

Référence : Karp, R. M. (1972). Reducibility among combinatorial problems. In Complexity of computer computations (pp. 85-103). Springer, Boston, MA.[$^{10}$](#52-références-scientifiques)

## 3.2. Preuve de la complexité du TSP par réduction polynomiale

Le problème de décision du Voyageur de Commerce (TSP) consiste à savoir s'il existe une tournée visitant toutes les villes avec un coût total inférieur ou égal à une limite $L$. Nous pouvons démontrer que le TSP est au moins aussi difficile que le Cycle Hamiltonien par une réduction polynomiale ($HC \leq_p TSP$). Soit une instance du problème du Cycle Hamiltonien définie par un graphe $G = (V, E)$. Nous construisons une instance du TSP de la manière suivante : Nous créons un graphe complet $G' = (V, E', c)$ ayant les mêmes sommets que $G$.Nous définissons la fonction de coût $c_{ij}$ pour chaque arête $(i, j)$ dans $E'$ :$$c_{ij} = \begin{cases} 1 & \text{si } (i, j) \in E \\ 2 & \text{si } (i, j) \notin E \end{cases}$$ Nous fixons notre limite de coût à $L = |V|$ (le nombre de sommets). Équivalence :Si $G$ possède un cycle hamiltonien, alors en suivant les mêmes arêtes dans $G'$, chaque arête aura un coût de $1$. Le coût total de la tournée sera exactement $|V|$. Inversement, si $G'$ possède une tournée de coût inférieur ou égal à $|V|$, cela signifie que la tournée n'emprunte que des arêtes de coût $1$ (puisqu'il y a $|V|$ arêtes dans la tournée). Donc, toutes ces arêtes existent dans le graphe d'origine $G$, formant ainsi un cycle hamiltonien.La transformation du graphe $G$ en $G'$ se fait en temps polynomial (en parcourant toutes les paires de sommets, soit $O(|V|^2)$). Le problème HC se réduisant polynomialement au problème TSP, le TSP est donc NP-complet.

## 3.3. Application à notre problème d'optimisation

Notre modélisation, qui répond aux exigences de l'ADEME, inclut des restrictions d'arêtes et des contraintes de précédence. Il s'agit d'une variante du problème de voyageur du commerce.Pour prouver que notre problème d'optimisation est NP-difficile, nous utilisons le principe de restriction : Considérons une instance de notre problème logistique où nous relâchons volontairement nos contraintes spécifiques : Aucune arête n'est interdite : $E_{interdit} = \emptyset$ Aucune dépendance n'est requise entre les villes : $P = \emptyset$ Dans ces conditions restrictives, notre problème devient strictement identique au TSP d'optimisation classique. Puisque notre problème général contient le TSP (qui est NP-difficile) comme un cas particulier, notre problème est par définition au moins aussi complexe que le TSP.

# 4 - Bibliographie

1. 
2. [**Problème de vendeur de voyages. (s. d.).** *Google For Developers.*](https://developers.google.com/optimization/routing/tsp?hl=fr) - visité le 09/04/2026
3. [**Problème d’itinéraire. (s. d.).** *Google For Developers.*](https://developers.google.com/optimization/routing/vrp?hl=fr) - visité le 09/04/2026
4. 
5. [**Dantzig, G. B., & Ramser, J. H. (1959).** *The Truck Dispatching Problem*. Management Science, 6(1), 80-91.](https://pubsonline.informs.org/doi/10.1287/mnsc.6.1.80)
6. [**Miller, C. E., Tucker, A. W., & Zemlin, R. A. (1960).** *Integer Programming Formulation of Traveling Salesman Problems*. Journal of the ACM (JACM), 7(4), 326-329.](https://dl.acm.org/doi/10.1145/321043.321046)
7. [**Diego. (2020, 24 juin).** *Traveling Salesman Problem (TSP) with Miller-Tucker-Zemlin (MTZ) in CPLEX/OPL – CoEnzyme.*](https://co-enzyme.fr/blog/traveling-salesman-problem-tsp-in-cplex-opl-with-miller-tucker-zemlin-mtz-formulation/)
8. [**Teaching Integer Programming Formulations Using the Traveling Salesman Problem. (2003).** *SIAM REVIEW, 4.*](https://mate.unipv.it/~gualandi/famo2conti/papers/teaching_tsp.pdf)
9. [**Garey, M. R., & Johnson, D. S. (1979).** *Computers and Intractability: A Guide to the Theory of NP-Completeness. W. H. Freeman & Co.*](https://perso.limos.fr/~palafour/PAPERS/PDF/Garey-Johnson79.pdf)
10. [**Karp, R. M. (1972).** *Reducibility among combinatorial problems*. In Complexity of computer computations (pp. 85-103). Springer, Boston, MA.](https://www.cs.umd.edu/~gasarch/BLOGPAPERS/Karp.pdf)

# 5 - Méthodes de résolution